# National Parks Brochure Scraper

This notebook scrapes brochures from npshistory.com, downloads PDFs, extracts information, and writes results to Google Sheets.

## Features:
- Scrapes brochure links from all US National Parks
- Downloads up to 20 PDF files (configurable for testing)
- Extracts text from each PDF
- Parses: park name, state, established year, size
- Writes results to Google Sheets
- Respects 10-second delay between requests
- Handles errors gracefully

## Instructions:
1. Run each cell in order
2. Authenticate when prompted for Google Sheets access
3. Results will be saved to both JSON file and Google Sheets

## Step 1: Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 PyPDF2 gspread google-auth
print("✓ Dependencies installed successfully!")

## Step 2: Import Libraries and Define Scraper Class

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import os
from urllib.parse import urljoin, urlparse
import PyPDF2
from datetime import datetime
import json
import gspread
from google.colab import auth
from google.auth import default

print("✓ Libraries imported successfully!")

In [ ]:
class NationalParksScraper:
    """Scraper for National Parks brochures from npshistory.com"""

    # Comprehensive list of US National Parks with their 4-letter codes
    NATIONAL_PARKS = {
        'ACAD': {'name': 'Acadia National Park', 'state': 'Maine'},
        'ARCH': {'name': 'Arches National Park', 'state': 'Utah'},
        'BADL': {'name': 'Badlands National Park', 'state': 'South Dakota'},
        'BIBE': {'name': 'Big Bend National Park', 'state': 'Texas'},
        'BISC': {'name': 'Biscayne National Park', 'state': 'Florida'},
        'BLCA': {'name': 'Black Canyon of the Gunnison National Park', 'state': 'Colorado'},
        'BRCA': {'name': 'Bryce Canyon National Park', 'state': 'Utah'},
        'CANY': {'name': 'Canyonlands National Park', 'state': 'Utah'},
        'CARE': {'name': 'Capitol Reef National Park', 'state': 'Utah'},
        'CAVE': {'name': 'Carlsbad Caverns National Park', 'state': 'New Mexico'},
        'CHIS': {'name': 'Channel Islands National Park', 'state': 'California'},
        'CONG': {'name': 'Congaree National Park', 'state': 'South Carolina'},
        'CRLA': {'name': 'Crater Lake National Park', 'state': 'Oregon'},
        'CUVA': {'name': 'Cuyahoga Valley National Park', 'state': 'Ohio'},
        'DENA': {'name': 'Denali National Park', 'state': 'Alaska'},
        'DRTO': {'name': 'Dry Tortugas National Park', 'state': 'Florida'},
        'EVER': {'name': 'Everglades National Park', 'state': 'Florida'},
        'GAAR': {'name': 'Gates of the Arctic National Park', 'state': 'Alaska'},
        'GLAC': {'name': 'Glacier National Park', 'state': 'Montana'},
        'GLBA': {'name': 'Glacier Bay National Park', 'state': 'Alaska'},
        'GRBA': {'name': 'Great Basin National Park', 'state': 'Nevada'},
        'GRCA': {'name': 'Grand Canyon National Park', 'state': 'Arizona'},
        'GRSA': {'name': 'Great Sand Dunes National Park', 'state': 'Colorado'},
        'GRSM': {'name': 'Great Smoky Mountains National Park', 'state': 'Tennessee/North Carolina'},
        'GRTE': {'name': 'Grand Teton National Park', 'state': 'Wyoming'},
        'GUMO': {'name': 'Guadalupe Mountains National Park', 'state': 'Texas'},
        'HALE': {'name': 'Haleakalā National Park', 'state': 'Hawaii'},
        'HAVO': {'name': 'Hawaiʻi Volcanoes National Park', 'state': 'Hawaii'},
        'HOSP': {'name': 'Hot Springs National Park', 'state': 'Arkansas'},
        'ISRO': {'name': 'Isle Royale National Park', 'state': 'Michigan'},
        'JOTR': {'name': 'Joshua Tree National Park', 'state': 'California'},
        'KATM': {'name': 'Katmai National Park', 'state': 'Alaska'},
        'KEFJ': {'name': 'Kenai Fjords National Park', 'state': 'Alaska'},
        'KOVA': {'name': 'Kobuk Valley National Park', 'state': 'Alaska'},
        'LACL': {'name': 'Lake Clark National Park', 'state': 'Alaska'},
        'LAVO': {'name': 'Lassen Volcanic National Park', 'state': 'California'},
        'MACA': {'name': 'Mammoth Cave National Park', 'state': 'Kentucky'},
        'MEVE': {'name': 'Mesa Verde National Park', 'state': 'Colorado'},
        'MORA': {'name': 'Mount Rainier National Park', 'state': 'Washington'},
        'NOCA': {'name': 'North Cascades National Park', 'state': 'Washington'},
        'OLYM': {'name': 'Olympic National Park', 'state': 'Washington'},
        'PEFO': {'name': 'Petrified Forest National Park', 'state': 'Arizona'},
        'PINN': {'name': 'Pinnacles National Park', 'state': 'California'},
        'REDW': {'name': 'Redwood National Park', 'state': 'California'},
        'ROMO': {'name': 'Rocky Mountain National Park', 'state': 'Colorado'},
        'SAGU': {'name': 'Saguaro National Park', 'state': 'Arizona'},
        'SEKI': {'name': 'Sequoia National Park', 'state': 'California'},
        'SHEN': {'name': 'Shenandoah National Park', 'state': 'Virginia'},
        'THRO': {'name': 'Theodore Roosevelt National Park', 'state': 'North Dakota'},
        'VOYA': {'name': 'Voyageurs National Park', 'state': 'Minnesota'},
        'WICA': {'name': 'Wind Cave National Park', 'state': 'South Dakota'},
        'WRST': {'name': 'Wrangell-St. Elias National Park', 'state': 'Alaska'},
        'YELL': {'name': 'Yellowstone National Park', 'state': 'Wyoming/Montana/Idaho'},
        'YOSE': {'name': 'Yosemite National Park', 'state': 'California'},
        'ZION': {'name': 'Zion National Park', 'state': 'Utah'},
        'INDU': {'name': 'Indiana Dunes National Park', 'state': 'Indiana'},
        'GOGA': {'name': 'Gateway Arch National Park', 'state': 'Missouri'},
        'NPSA': {'name': 'New River Gorge National Park', 'state': 'West Virginia'},
        'WHSA': {'name': 'White Sands National Park', 'state': 'New Mexico'},
    }

    def __init__(self, base_url="https://npshistory.com/publications/",
                 download_dir="national_parks_brochures",
                 request_delay=10,
                 max_pdfs=20):
        self.base_url = base_url
        self.download_dir = download_dir
        self.request_delay = request_delay
        self.max_pdfs = max_pdfs
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
        })
        os.makedirs(self.download_dir, exist_ok=True)
        self.results = []
        self.errors = []

    def get_page(self, url, retries=3):
        for attempt in range(retries):
            try:
                print(f"Fetching: {url} (attempt {attempt + 1}/{retries})")
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                return response
            except requests.RequestException as e:
                print(f"Error fetching {url}: {e}")
                if attempt < retries - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    self.errors.append({'url': url, 'error': str(e), 'type': 'fetch_error'})
                    return None
        return None

    def scrape_park_brochures(self, park_code):
        park_code_lower = park_code.lower()
        brochure_url = f"{self.base_url}{park_code_lower}/brochures/index.htm"
        print(f"\nScraping brochures for {park_code} - {self.NATIONAL_PARKS.get(park_code, {}).get('name', 'Unknown')}")
        response = self.get_page(brochure_url)
        if not response:
            return []
        soup = BeautifulSoup(response.content, 'html.parser')
        brochure_links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.endswith('.pdf'):
                pdf_url = urljoin(brochure_url, href)
                brochure_links.append(pdf_url)
        print(f"Found {len(brochure_links)} brochure links")
        return brochure_links

    def download_pdf(self, url, filename):
        try:
            print(f"Downloading: {url}")
            response = self.session.get(url, timeout=60)
            response.raise_for_status()
            filepath = os.path.join(self.download_dir, filename)
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"✓ Saved: {filepath}")
            return filepath
        except Exception as e:
            print(f"✗ Error downloading {url}: {e}")
            self.errors.append({'url': url, 'error': str(e), 'type': 'download_error'})
            return None

    def extract_text_from_pdf(self, pdf_path):
        try:
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text()
                return text
        except Exception as e:
            print(f"✗ Error extracting text from {pdf_path}: {e}")
            self.errors.append({'file': pdf_path, 'error': str(e), 'type': 'extraction_error'})
            return ""

    def parse_park_info(self, text, park_code):
        info = {
            'park_name': self.NATIONAL_PARKS.get(park_code, {}).get('name', 'Unknown'),
            'state': self.NATIONAL_PARKS.get(park_code, {}).get('state', 'Unknown'),
            'established_year': 'Unknown',
            'size': 'Unknown'
        }
        year_patterns = [
            r'established[:\s]+(\d{4})',
            r'est[.\s]+(\d{4})',
            r'designated[:\s]+(\d{4})',
            r'authorized[:\s]+(\d{4})',
            r'created[:\s]+(\d{4})',
        ]
        for pattern in year_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                info['established_year'] = match.group(1)
                break
        size_patterns = [
            r'([\d,]+)\s*acres',
            r'([\d,]+)\s*square miles',
            r'([\d,]+)\s*sq\.\s*mi',
        ]
        for pattern in size_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                info['size'] = match.group(0)
                break
        return info

    def scrape_all_parks(self):
        pdf_count = 0
        for park_code in sorted(self.NATIONAL_PARKS.keys()):
            if pdf_count >= self.max_pdfs:
                print(f"\n✓ Reached maximum PDF limit ({self.max_pdfs}). Stopping.")
                break
            brochure_links = self.scrape_park_brochures(park_code)
            time.sleep(self.request_delay)
            for pdf_url in brochure_links:
                if pdf_count >= self.max_pdfs:
                    break
                filename = f"{park_code}_{os.path.basename(urlparse(pdf_url).path)}"
                pdf_path = self.download_pdf(pdf_url, filename)
                if pdf_path:
                    text = self.extract_text_from_pdf(pdf_path)
                    park_info = self.parse_park_info(text, park_code)
                    park_info['pdf_url'] = pdf_url
                    park_info['local_file'] = pdf_path
                    park_info['park_code'] = park_code
                    park_info['scraped_at'] = datetime.now().isoformat()
                    self.results.append(park_info)
                    pdf_count += 1
                    print(f"\n[{pdf_count}/{self.max_pdfs}] Processed: {park_info['park_name']}")
                    print(f"  State: {park_info['state']} | Established: {park_info['established_year']} | Size: {park_info['size']}")
                if pdf_count < self.max_pdfs:
                    print(f"\n⏳ Waiting {self.request_delay} seconds before next request...")
                    time.sleep(self.request_delay)
        return self.results

    def save_to_json(self, filename='national_parks_results.json'):
        output = {
            'results': self.results,
            'errors': self.errors,
            'metadata': {
                'total_parks_processed': len(set([r['park_code'] for r in self.results])),
                'total_pdfs_downloaded': len(self.results),
                'total_errors': len(self.errors),
                'scrape_date': datetime.now().isoformat()
            }
        }
        filepath = os.path.join(self.download_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(output, f, indent=2)
        print(f"\n✓ Results saved to: {filepath}")
        return filepath

    def write_to_google_sheets(self, spreadsheet_name='National Parks Brochures'):
        try:
            auth.authenticate_user()
            creds, _ = default()
            gc = gspread.authorize(creds)
            try:
                spreadsheet = gc.open(spreadsheet_name)
                worksheet = spreadsheet.sheet1
                worksheet.clear()
            except gspread.SpreadsheetNotFound:
                spreadsheet = gc.create(spreadsheet_name)
                worksheet = spreadsheet.sheet1
            headers = ['Park Code', 'Park Name', 'State', 'Established Year',
                      'Size', 'PDF URL', 'Local File', 'Scraped At']
            rows = [headers]
            for result in self.results:
                row = [
                    result.get('park_code', ''),
                    result.get('park_name', ''),
                    result.get('state', ''),
                    result.get('established_year', ''),
                    result.get('size', ''),
                    result.get('pdf_url', ''),
                    result.get('local_file', ''),
                    result.get('scraped_at', '')
                ]
                rows.append(row)
            worksheet.update('A1', rows)
            print(f"\n✓ Results written to Google Sheet: {spreadsheet_name}")
            print(f"📊 URL: {spreadsheet.url}")
            return spreadsheet.url
        except Exception as e:
            print(f"✗ Error writing to Google Sheets: {e}")
            self.errors.append({'error': str(e), 'type': 'sheets_error'})
            return None

    def print_summary(self):
        print("\n" + "="*70)
        print("SCRAPING SUMMARY")
        print("="*70)
        print(f"Total Parks Processed: {len(set([r['park_code'] for r in self.results]))}")
        print(f"Total PDFs Downloaded: {len(self.results)}")
        print(f"Total Errors: {len(self.errors)}")

print("✓ Scraper class defined successfully!")

## Step 3: Configure and Run Scraper

In [ ]:
# Configure scraper
scraper = NationalParksScraper(
    download_dir='national_parks_brochures',
    request_delay=10,  # 10 seconds between requests
    max_pdfs=20  # Limit to 20 PDFs for testing
)

print("="*70)
print("STARTING NATIONAL PARKS BROCHURE SCRAPER")
print("="*70)
print(f"Rate limit: {scraper.request_delay} seconds between requests")
print(f"Maximum PDFs: {scraper.max_pdfs}")
print(f"Total parks to check: {len(scraper.NATIONAL_PARKS)}")
print("="*70)

# Run the scraper
results = scraper.scrape_all_parks()

print("\n✓ Scraping complete!")

## Step 4: Save Results to JSON

In [ ]:
# Save to JSON
json_path = scraper.save_to_json()

# Display some results
print("\n" + "-"*70)
print("SAMPLE RESULTS:")
print("-"*70)
for i, result in enumerate(results[:5], 1):
    print(f"\n{i}. {result['park_name']}")
    print(f"   State: {result['state']}")
    print(f"   Established: {result['established_year']}")
    print(f"   Size: {result['size']}")
    print(f"   PDF: {result['pdf_url']}")

## Step 5: Write Results to Google Sheets

**Note:** You will be prompted to authenticate with Google. Click the link and follow the instructions.

In [ ]:
# Write to Google Sheets
sheet_url = scraper.write_to_google_sheets('National Parks Brochures')

if sheet_url:
    print(f"\n🎉 Success! Open your Google Sheet here:")
    print(sheet_url)
else:
    print("\n⚠️ Could not create Google Sheet. Check errors above.")

## Step 6: View Summary

In [ ]:
# Print detailed summary
scraper.print_summary()

# Show errors if any
if scraper.errors:
    print("\n" + "-"*70)
    print("ERRORS ENCOUNTERED:")
    print("-"*70)
    for i, error in enumerate(scraper.errors[:10], 1):
        print(f"{i}. {error.get('type', 'unknown')}: {str(error.get('error', ''))[:100]}")

## Step 7: Download Files (Optional)

Download the results and PDFs to your local machine.

In [ ]:
from google.colab import files
import shutil

# Create a zip file of all results
shutil.make_archive('national_parks_results', 'zip', 'national_parks_brochures')

# Download the zip file
print("Downloading results...")
files.download('national_parks_results.zip')
print("✓ Download complete!")

## Notes

### Customization Options:

1. **Change PDF limit**: Modify `max_pdfs` parameter (e.g., `max_pdfs=50`)
2. **Adjust delay**: Modify `request_delay` parameter (e.g., `request_delay=15`)
3. **Target specific parks**: Modify the scraper to only check certain park codes

### Troubleshooting:

- If you get 403 errors, the website may be blocking requests. Try increasing the delay.
- If PDF extraction fails, some PDFs may be scanned images without text.
- If Google Sheets authentication fails, make sure you're running in Google Colab.

### Data Quality:

- Not all brochures contain structured data about established year and size
- The parser uses regex patterns to extract information
- Some fields may show 'Unknown' if the information isn't found in the PDF